# 🎬 MatAnyone 2 — Video Matting
**Istruzioni:**
1. Vai su `Runtime > Change runtime type` → seleziona **T4 GPU**
2. Esegui le celle in ordine (▶ su ognuna, oppure `Runtime > Run all`)
3. Nella cella 4 carica il tuo **video** e la **maschera PNG** (primo frame del soggetto)
4. Scarica i risultati dall'ultima cella

In [ ]:
#@title ⚙️ 1. Installa dipendenze
!pip install -q git+https://github.com/pq-yang/MatAnyone2.git
!apt-get install -q ffmpeg
print('✅ Installazione completata')

In [ ]:
#@title 📦 2. Scarica il modello
from matanyone2 import MatAnyone2, InferenceCore
model = MatAnyone2.from_pretrained('PeiqingYang/MatAnyone2')
processor = InferenceCore(model, device='cuda:0')
print('✅ Modello caricato')

In [ ]:
#@title 📁 3. Crea cartelle di lavoro
import os
os.makedirs('inputs/video', exist_ok=True)
os.makedirs('inputs/mask', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('✅ Cartelle pronte')

In [ ]:
#@title ⬆️ 4. Carica il tuo VIDEO e la MASCHERA PNG
from google.colab import files

print('--- Carica il VIDEO (mp4, mov, avi) ---')
uploaded_video = files.upload()
video_filename = list(uploaded_video.keys())[0]
import shutil
shutil.move(video_filename, f'inputs/video/{video_filename}')
VIDEO_PATH = f'inputs/video/{video_filename}'
print(f'✅ Video caricato: {VIDEO_PATH}')

print('\n--- Carica la MASCHERA del primo frame (PNG bianco/nero) ---')
print('   (bianco = soggetto, nero = sfondo)')
uploaded_mask = files.upload()
mask_filename = list(uploaded_mask.keys())[0]
shutil.move(mask_filename, f'inputs/mask/{mask_filename}')
MASK_PATH = f'inputs/mask/{mask_filename}'
print(f'✅ Maschera caricata: {MASK_PATH}')

In [ ]:
#@title 🎨 5. (Opzionale) Non hai la maschera? Creala automaticamente con SAM2
# Salta questa cella se hai già caricato la maschera sopra

!pip install -q segment-anything-2

import torch
from PIL import Image
import numpy as np
import cv2

# Estrai il primo frame dal video
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()
first_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
Image.fromarray(first_frame).save('inputs/first_frame.png')

print('✅ Primo frame estratto: inputs/first_frame.png')
print('Aprilo, crea una maschera manualmente e caricala nella cella 4')

from IPython.display import display
display(Image.fromarray(first_frame))

In [ ]:
#@title 🚀 6. Esegui il matting
print('Avvio MatAnyone 2... (può richiedere qualche minuto)')

foreground_path, alpha_path = processor.process_video(
    input_path=VIDEO_PATH,
    mask_path=MASK_PATH,
    output_path='outputs',
)

print(f'\n✅ Completato!')
print(f'   Foreground (RGB): {foreground_path}')
print(f'   Alpha (grayscale): {alpha_path}')

In [ ]:
#@title 🎞️ 7. (Bonus) Genera anche il WebM con alpha trasparente
import subprocess

webm_output = 'outputs/output_alpha.webm'

# Combina RGB + alpha in un unico WebM VP9 con canale alpha
cmd = [
    'ffmpeg', '-y',
    '-i', foreground_path,
    '-i', alpha_path,
    '-filter_complex',
    '[0:v]format=rgba[fg];[1:v]format=gray[a];[fg][a]alphamerge[out]',
    '-map', '[out]',
    '-c:v', 'libvpx-vp9',
    '-pix_fmt', 'yuva420p',
    '-b:v', '0', '-crf', '18',
    webm_output
]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ WebM con alpha generato: {webm_output}')
else:
    print('⚠️ Errore FFmpeg:')
    print(result.stderr[-2000:])

In [ ]:
#@title ⬇️ 8. Scarica i risultati
from google.colab import files
import os

print('File disponibili:')
for f in os.listdir('outputs'):
    size = os.path.getsize(f'outputs/{f}') / 1024
    print(f'  📄 {f} ({size:.0f} KB)')

print('\nScarico tutti i file...')
for f in os.listdir('outputs'):
    files.download(f'outputs/{f}')

print('✅ Download avviato')